# Heart Disease EDA and Modeling

This notebook demonstrates:

1. Data loading and cleaning
2. Exploratory visualizations
3. Baseline model comparison

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_loader import HEART_COLUMNS, preprocess_data

sns.set_theme(style="whitegrid")

df = pd.read_csv("data/raw/heart_disease.csv", header=None, names=HEART_COLUMNS)
clean_df = preprocess_data(df)
clean_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(clean_df["age"], kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.countplot(x="target", data=clean_df, ax=axes[1])
axes[1].set_title("Class Balance")

corr = clean_df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, ax=axes[2])
axes[2].set_title("Correlation Heatmap")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.pipeline import build_training_pipeline, NUMERIC_FEATURES, CATEGORICAL_FEATURES

X = clean_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = clean_df["target"]

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, solver="liblinear"),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}

for name, estimator in models.items():
    pipe = build_training_pipeline(estimator)
    scores = cross_val_score(pipe, X, y, cv=5, scoring=make_scorer(roc_auc_score, response_method="predict_proba"))
    print(f"{name} ROC-AUC: {scores.mean():.4f} +/- {scores.std():.4f}")